# Monitoring & Logging - IMDB BiLSTM
**Kriteria 4** - Dicoding Membangun Sistem Machine Learning

Jalankan notebook ini di **Google Colab** untuk serving model + exporter.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# GANTI PATH INI sesuai folder Dicoding-Training_Model di Drive Anda
PROJECT_DIR = "/content/drive/MyDrive/Dicoding-Training_Model"
MODEL_DIR = f"{PROJECT_DIR}/Membangun_model"
MONITOR_DIR = f"{PROJECT_DIR}/Monitoring_dan_Logging"
print(f"Model dir: {MODEL_DIR}")
print(f"Monitor dir: {MONITOR_DIR}")

In [ ]:
# Install dependencies
!pip install -q mlflow tensorflow scikit-learn numpy matplotlib requests prometheus_client pandas

---
## Step 1: Serve Model dengan MLflow

In [ ]:
import os
import glob

# Cari path artifacts MLflow
artifacts_paths = glob.glob(f"{MODEL_DIR}/mlruns/*/models/*/artifacts/MLmodel")
for p in artifacts_paths:
    print(p)

# Ambil path artifacts pertama
if artifacts_paths:
    ARTIFACTS_DIR = os.path.dirname(artifacts_paths[0])
    print(f"\nMenggunakan: {ARTIFACTS_DIR}")
else:
    print("Tidak ada model ditemukan!")

In [ ]:
# Start MLflow serving di background
import subprocess
import time

log_file = open('/content/mlflow_serve.log', 'w')
serve_proc = subprocess.Popen(
    ['mlflow', 'models', 'serve', '-m', ARTIFACTS_DIR, '--port', '8080', '--no-conda'],
    stdout=log_file, stderr=log_file
)
print(f"MLflow serve PID: {serve_proc.pid}")

# Tunggu sampai server siap
for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8080/ping", timeout=2)
        if r.status_code == 200:
            print(f"Server siap setelah {i+1}s")
            break
    except:
        pass
    time.sleep(1)
else:
    print("Server tidak siap. Cek log:")
    !cat /content/mlflow_serve.log

---
## Step 2: Uji Inference & Screenshot

In [ ]:
import requests
import json
import numpy as np
import pickle

# Load data
X_test = np.load(f"{MODEL_DIR}/dataset_preprocessing/X_test.npy")
y_test = np.load(f"{MODEL_DIR}/dataset_preprocessing/y_test.npy")
with open(f"{MODEL_DIR}/dataset_preprocessing/label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

# Test inference
sample = X_test[:5]
payload = {"instances": sample.tolist()}

response = requests.post(
    "http://127.0.0.1:8080/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    preds = np.argmax(result["predictions"], axis=1)
    actual_labels = encoder.inverse_transform(y_test[:5])
    pred_labels = encoder.inverse_transform(preds)
    for i in range(5):
        print(f"{i}: Pred={pred_labels[i]:>7} | Actual={actual_labels[i]:>7} | {'✅' if preds[i]==y_test[i] else '❌'}")
else:
    print(response.text)

**📸 Screenshot hasil di atas** → simpan sebagai `1.bukti_serving/bukti_serving.png`

---
## Step 3: Jalankan Prometheus Exporter

In [ ]:
# Copy exporter ke Colab dan jalankan
import shutil
shutil.copy(f"{MONITOR_DIR}/3.prometheus_exporter.py", "/content/3.prometheus_exporter.py")

# Adjust DATA_PATH di exporter untuk Colab
!sed -i 's|DATA_DIR = "\.\./Membangun_model/dataset_preprocessing"|DATA_DIR = "{MODEL_DIR}/dataset_preprocessing"|' /content/3.prometheus_exporter.py

exporter_log = open('/content/exporter.log', 'w')
exporter_proc = subprocess.Popen(
    ['python', '/content/3.prometheus_exporter.py'],
    stdout=exporter_log, stderr=exporter_log
)
print(f"Exporter PID: {exporter_proc.pid}")
print("Exporter berjalan di port 8001 (metrics: /metrics)")

In [ ]:
# Cek metrik exporter
import time
time.sleep(15)
r = requests.get("http://127.0.0.1:8001/metrics")
print(r.text[:1000])

---
## Step 4: Prometheus & Grafana (via Docker)

Jalankan perintah berikut di **terminal lokal** (bukan Colab):

### Prometheus
```bash
docker run -p 9090:9090 \
  -v "$(pwd)/Monitoring_dan_Logging/2.prometheus.yml:/etc/prometheus/prometheus.yml" \
  prom/prometheus
```
Buka http://localhost:9090

### Grafana
```bash
docker run -d -p 3000:3000 --name grafana grafana/grafana
```
Buka http://localhost:3000 (admin/admin)

### Setup Grafana
1. Add datasource → Prometheus → URL: `http://host.docker.internal:9090`
2. Import dashboard atau buat panel untuk 5 metrik:
   - `model_predictions_total`
   - `model_prediction_latency_seconds`
   - `model_endpoint_up`
   - `model_prediction_error_rate`
   - `model_accuracy_score`

### Alerting
1. Alerting → Alert rules → New alert rule
2. Condition: `model_prediction_error_rate > 0.5`
3. Set folder + evaluation group
4. Screenshot rules + notifikasi

---
## Step 5: Screenshot Semua Bukti

| Folder | Isi |
|--------|-----|
| `1.bukti_serving/` | Hasil response inference (dari Step 2) |
| `4.bukti monitoring Prometheus/` | 5 screenshot metrik di Prometheus |
| `5.bukti monitoring Grafana/` | 5 screenshot panel Grafana |
| `6.bukti alerting Grafana/` | Screenshot rules & notifikasi |

In [ ]:
# Matikan server setelah selesai
serve_proc.kill()
exporter_proc.kill()
print("Server dan exporter dihentikan.")